# Fine-tune XTTS-v2 on the Sinhala radio-drama dataset

Trains [DSEgrp18/XTTS_V2_Baseline](https://github.com/DSEgrp18/XTTS_V2_Baseline)
on the clips built by `kaggle_build_dataset.py`.

## Settings — different from the dataset notebook

| Setting | Value |
|---|---|
| **Accelerator** | **GPU T4 x2** (or P100) — training is GPU-bound and will not run without one |
| **Internet** | **ON** (clones the repo, downloads the XTTS base) |
| **Data** | attach your dataset output (the one containing `dataset/all_manifests.csv`) |

## Two things to know before you start

**XTTS drops anything over 11.6 s.** `GPTArgs.max_wav_length` is 255995 samples at
22050 Hz. Our clips run to 30 s, and the dataloader discards the long ones
*silently*. The filter cell below removes them where you can see the count —
about 16% of the audio.

**Speaker identity will not work.** Radio drama is multi-speaker and nothing
diarizes it, so every clip trains under one speaker label. XTTS conditions on a
reference clip during training; if that "speaker" is thirty actors, the model
learns the reference does not predict the output voice. Expect this run to teach
Sinhala phonetics and prosody, **not** controllable voice cloning. Diarization is
the fix, and it belongs in the dataset build rather than here.

## Order
`filter -> prepare -> base model -> extend vocab -> smoke -> train -> monitor -> export -> infer`

`extend_vocab` matters most: XTTS-v2 has no Sinhala in its BPE vocabulary, so
without it every Sinhala character becomes `[UNK]` and the model trains on
"unknown unknown unknown". The loss falls; the output is babble.


## 1. Install
Restart the session after this cell.

In [ ]:
# coqui-tts is the maintained idiap fork. Do NOT `pip install TTS` -- that one
# pins torch<2.1 and replaces Kaggle's CUDA build with a CPU wheel.
!pip install -q "coqui-tts>=0.25.1" "coqui-tts-trainer>=0.2.0" librosa soundfile tensorboard huggingface_hub

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("")
    print("*** NO GPU. Settings -> Accelerator -> GPU T4 x2, then restart. ***")
print("")
print(">>> Now: Run -> Restart session, then continue from the NEXT cell. <<<")


## 2. Clone the repo, locate the dataset

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/DSEgrp18/XTTS_V2_Baseline.git"
CODE_DIR = "/kaggle/working/XTTS_V2_Baseline"
if not os.path.isdir(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=False)
SRC = CODE_DIR + "/src"

# Find our dataset wherever Kaggle mounted it. Depth differs between a notebook
# output and an uploaded dataset, so search instead of assuming a layout.
OUR_DATA = None
inp = pathlib.Path("/kaggle/input")
if inp.is_dir():
    hits = sorted(inp.rglob("all_manifests.csv"))
    if hits:
        OUR_DATA = str(hits[0].parent)
    else:
        mans = sorted(inp.rglob("manifest.csv"))
        if mans:
            OUR_DATA = str(mans[0].parent.parent)
print("code    :", CODE_DIR)
print("dataset :", OUR_DATA or "NOT FOUND -> Add Data, then re-run this cell")

DATASET    = "/kaggle/working/xtts_data"
XTTS_BASE  = "/kaggle/working/xtts_base"
XTTS_SI    = "/kaggle/working/xtts_si"
RUN_DIR    = "/kaggle/working/run"
EXPORT_DIR = "/kaggle/working/export"
META       = "/kaggle/working/xtts_metadata.csv"


In [ ]:
# coqui-tts moved XttsAudioConfig out of gpt_trainer and into
# TTS.tts.models.xtts, so the upstream import raises ImportError on current
# versions. Patch the clone to try both, which works on old and new alike.
import pathlib

_p = pathlib.Path(SRC) / "train_xtts.py"
_s = _p.read_text(encoding="utf-8")
_old = """    from TTS.tts.layers.xtts.trainer.gpt_trainer import (
        GPTArgs,
        GPTTrainer,
        GPTTrainerConfig,
        XttsAudioConfig,
    )"""
_new = """    from TTS.tts.layers.xtts.trainer.gpt_trainer import (
        GPTArgs,
        GPTTrainer,
        GPTTrainerConfig,
    )
    try:
        from TTS.tts.layers.xtts.trainer.gpt_trainer import XttsAudioConfig
    except ImportError:
        from TTS.tts.models.xtts import XttsAudioConfig"""

if _old in _s:
    _p.write_text(_s.replace(_old, _new), encoding="utf-8")
    print("patched train_xtts.py (XttsAudioConfig import)")
elif "from TTS.tts.models.xtts import XttsAudioConfig" in _s:
    print("train_xtts.py already patched")
else:
    print("WARNING: import block not found -- upstream may have fixed it; check manually")


## 3. Filter our clips to what XTTS can use

In [ ]:
%%writefile /kaggle/working/prepare_for_xtts.py
#!/usr/bin/env python3
"""
prepare_for_xtts.py -- turn this repo's manifest into input for XTTS_V2_Baseline.

    dataset/all_manifests.csv  ->  xtts_metadata.csv  (audio_file|text|speaker_name)

That file feeds DSEgrp18/XTTS_V2_Baseline's prepare_dataset.py, which does the
resampling, trimming and train/eval split. This script only decides WHICH clips
go in, because that repo's preparer knows nothing about our `quality` column or
about XTTS's hard length ceiling.

THREE FILTERS, and the second one is not optional
-------------------------------------------------
1. QUALITY   The corpus has two tiers. The 201-episode serial is 11 kHz / 16 kbps,
             so its clips are band-limited to 5.5 kHz -- fricatives are largely
             gone. Training voice quality on that teaches the model to generate
             muffled speech. Default is hifi-only; --quality all to include both.

2. LENGTH    XTTS GPTArgs.max_wav_length is 255995 samples @22050 Hz = 11.61 s.
             Longer clips are dropped SILENTLY by the dataloader. Our clips run
             to 30 s, so ~16% of the audio would vanish without warning. They are
             dropped here instead, where the count is printed.

3. SNAPPED   Clips whose timestamps were never confirmed against the VAD
             (`snapped=False`) can start or end mid-word. Excluded by default;
             --allow-unsnapped keeps them.

SPEAKER LABELS -- READ THIS
---------------------------
Radio drama is multi-speaker and nothing in this pipeline diarizes it, so every
clip is written under ONE speaker name. XTTS is a voice-cloning model: during
training it samples another clip from the same speaker as the conditioning
reference. If that "speaker" is actually thirty different actors, the model
learns that the reference clip does NOT predict the output voice -- which is
precisely the capability being fine-tuned.

Expect this run to teach Sinhala phonetics and prosody, and NOT to give
controllable voice identity. Speaker diarization is the fix, and it belongs
upstream in the dataset build.

USAGE
    python prepare_for_xtts.py --src dataset --out xtts_metadata.csv
    python prepare_for_xtts.py --src dataset --quality all --max-seconds 11.4
"""

from __future__ import annotations

import argparse
import csv
import json
import statistics as st
import sys
from collections import Counter
from pathlib import Path

# GPTArgs.max_wav_length = 255995 @ 22050 Hz = 11.61 s. Stay under it.
XTTS_MAX_SECONDS = 11.4
XTTS_MIN_SECONDS = 0.9
XTTS_MAX_CHARS = 190       # GPTArgs.max_text_length = 200 tokens; chars ~ tokens


def load_rows(src: Path) -> list[dict]:
    """Read all_manifests.csv, or every per-episode manifest if it is absent."""
    combined = src / "all_manifests.csv"
    if combined.exists():
        with combined.open(encoding="utf-8-sig") as fh:
            return list(csv.DictReader(fh))
    rows: list[dict] = []
    for man in sorted(src.glob("*/manifest.csv")):
        with man.open(encoding="utf-8-sig") as fh:
            rows.extend(csv.DictReader(fh))
    return rows


def main() -> int:
    ap = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--src", required=True,
                    help="dataset root holding <episode>/ dirs + all_manifests.csv")
    ap.add_argument("--out", default="xtts_metadata.csv")
    ap.add_argument("--quality", default="hifi", choices=["hifi", "lofi", "all"])
    ap.add_argument("--max-seconds", type=float, default=XTTS_MAX_SECONDS)
    ap.add_argument("--min-seconds", type=float, default=XTTS_MIN_SECONDS)
    ap.add_argument("--max-chars", type=int, default=XTTS_MAX_CHARS)
    ap.add_argument("--allow-unsnapped", action="store_true",
                    help="keep clips whose cut points were never VAD-verified")
    ap.add_argument("--speaker", default="muwan_palassa",
                    help="single speaker label (see the caveat in the docstring)")
    args = ap.parse_args()

    src = Path(args.src)
    rows = load_rows(src)
    if not rows:
        print(f"ERROR: no manifest under {src}", file=sys.stderr)
        return 2
    print(f"{len(rows)} clips in manifest")

    drop = Counter()
    kept: list[tuple[str, str]] = []
    for r in rows:
        q = r.get("quality", "hifi")
        if args.quality != "all" and q != args.quality:
            drop[f"quality!={args.quality}"] += 1
            continue
        if not args.allow_unsnapped and str(r.get("snapped", "True")) != "True":
            drop["unsnapped"] += 1
            continue
        try:
            dur = float(r["duration"])
        except (KeyError, ValueError):
            drop["bad_duration"] += 1
            continue
        if dur > args.max_seconds:
            drop[f"longer_than_{args.max_seconds:g}s"] += 1
            continue
        if dur < args.min_seconds:
            drop[f"shorter_than_{args.min_seconds:g}s"] += 1
            continue
        text = (r.get("text") or "").strip()
        if not text:
            drop["empty_text"] += 1
            continue
        if len(text) > args.max_chars:
            drop[f"text_over_{args.max_chars}_chars"] += 1
            continue
        kept.append((r["clip_id"], text))

    if not kept:
        print("ERROR: every clip was filtered out.", file=sys.stderr)
        print(json.dumps(drop, indent=2), file=sys.stderr)
        return 3

    # Pipe-separated with this exact header: XTTS_V2_Baseline/prepare_dataset.py
    # keys off "audio_file"/"text", and picks '|' as the delimiter when the
    # first line has more pipes than commas.
    out = Path(args.out)
    with out.open("w", encoding="utf-8", newline="") as fh:
        fh.write("audio_file|text|speaker_name\n")
        for cid, text in kept:
            fh.write(f"{cid}|{text}|{args.speaker}\n")

    # Report against the manifest's own durations, so the yield is visible
    # before an hour of GPU time is spent discovering it.
    by_id = {r["clip_id"]: r for r in rows}
    secs = [float(by_id[c]["duration"]) for c, _ in kept]
    total_h = sum(secs) / 3600
    print(f"\nwrote {out}  ({len(kept)} clips, {total_h:.2f} h)")
    print(f"  duration : median {st.median(secs):.2f}s  max {max(secs):.2f}s")
    print(f"  speaker  : {args.speaker!r} (single label -- see docstring)")
    if drop:
        print("\ndropped:")
        for reason, n in drop.most_common():
            print(f"  {n:6d}  {reason}")

    if total_h < 1.0:
        print("\nNOTE: under 1 hour. Enough to validate the pipeline end to end,"
              "\n      not enough to judge output quality. Keep building episodes.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# Filter our manifest to what XTTS can actually consume.
#   --quality hifi : 44.1 kHz episodes only (the 11 kHz serial is band-limited)
#   --quality all  : include the lo-fi serial too, once enough of it exists
!python /kaggle/working/prepare_for_xtts.py --src {OUR_DATA} --out {META} --quality hifi


## 4. Prepare the corpus (resample, trim, split)

In [ ]:
# Their preparer: resample to 22050, trim, normalise, split train/eval, and
# build the charset that extend_vocab needs.
!python {SRC}/prepare_dataset.py --src {OUR_DATA} --transcript {META} --out {DATASET} --speaker muwan_palassa --eval-size 128 --workers 4


In [ ]:
import json
report = json.load(open(DATASET + "/prepare_report.json", encoding="utf-8"))
print(json.dumps(report, ensure_ascii=False, indent=2))

cs = json.load(open(DATASET + "/charset.json", encoding="utf-8"))
print("")
print(str(len(cs["chars"])) + " unique characters")
print("".join(cs["chars"]))

# Eyeball the text. Silent corruption here is the top cause of a fine-tune that
# trains cleanly and then babbles.
print("")
for line in open(DATASET + "/metadata_train.csv", encoding="utf-8").readlines()[:6]:
    print(line.rstrip())


## 5. Base model, then Sinhala vocabulary

In [ ]:
!python {SRC}/download_base.py --out {XTTS_BASE}


In [ ]:
# Adds ~80 Sinhala characters plus a [si] token to the BPE vocab and grows the
# embedding matrices. Without this every Sinhala codepoint is [UNK].
!python {SRC}/extend_vocab.py --xtts-dir {XTTS_BASE} --charset {DATASET}/charset.json --out-dir {XTTS_SI} --min-count 3


## 6. Smoke test, then train

In [ ]:
# Tiny run to prove the wiring before committing GPU hours.
!python {SRC}/train_xtts.py --dataset {DATASET} --xtts-dir {XTTS_SI} --out /kaggle/working/smoke --smoke --batch-size 2


In [ ]:
!rm -rf /kaggle/working/smoke
!df -h /kaggle/working | tail -1


In [ ]:
# Real run, backgrounded so the notebook stays responsive.
import subprocess

EPOCHS, BATCH_SIZE, GRAD_ACCUM, LR = 12, 4, 21, 5e-6
LOG = "/kaggle/working/train.log"
cmd = ["python", SRC + "/train_xtts.py",
       "--dataset", DATASET, "--xtts-dir", XTTS_SI, "--out", RUN_DIR,
       "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
       "--grad-accum", str(GRAD_ACCUM), "--lr", str(LR),
       "--save-step", "2000", "--start-with-eval"]
print(" ".join(cmd))
with open(LOG, "w") as fh:
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT)
print("pid", proc.pid)


In [ ]:
!tail -n 40 /kaggle/working/train.log


## 7. Monitor for overfitting

In [ ]:
# Watch loss_mel_ce -- the acoustic term. loss_text_ce carries weight 0.01 and
# mostly reflects the new Sinhala embeddings settling in; its early drop is not
# progress on audio quality.
!python {SRC}/monitor.py --run {RUN_DIR}

from IPython.display import Image, display
import glob
for p in sorted(glob.glob(RUN_DIR + "/*/monitor/curves.png")):
    display(Image(filename=p))


## 8. Export at the eval minimum, then listen

In [ ]:
import glob, json, os
run = max(glob.glob(RUN_DIR + "/GPT_XTTS_si-*"), key=os.path.getmtime)
at_step = ""
vp = run + "/monitor/verdict.json"
if os.path.isfile(vp):
    v = json.load(open(vp))
    print(v["status"], "->", v["detail"])
    if v["status"] == "overfitting":
        at_step = "--at-step " + str(v["best_step"])   # export the eval minimum
print("exporting from", run, at_step)
!python {SRC}/export_checkpoint.py --run {run} --xtts-dir {XTTS_SI} --out {EXPORT_DIR} {at_step}


In [ ]:
import glob
REF = sorted(glob.glob(DATASET + "/wavs/*.wav"))[0]
print("reference:", REF)
!python {SRC}/infer.py --model-dir {EXPORT_DIR} --ref {REF} --out /kaggle/working/samples

from IPython.display import Audio, display
for p in sorted(glob.glob("/kaggle/working/samples/*.wav")):
    print(p)
    display(Audio(p))
